In [13]:
import pickle
import numpy as np
from scipy import signal
from sklearn.metrics import auc
from scipy.stats import pearsonr
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [7]:
with open('/home/emanu/Documents/PycharmProjects/HPEClinicalEvaluation/results/mediapipe-lower-None-0.pkl', 'rb') as file:
    data = pickle.load(file)
    
gt = data['gt_keypoints']
pred = data['pred_keypoints']
frame = data['frame']

model_name = 'mediapipe'

In [8]:
def plot_3d_keypoints_validation(keypoints_ground_truth, keypoints_hpe_truth, model_name):
    colors = ['red', 'blue']
    names = ['Ground Truth KeyPoints', 'HPE Truth KeyPoints']

    # Define connections between related keypoints
    if model_name == 'mediapipe':
        connections = [(0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (6, 7), (0, 6),
                       (1, 7), (6, 8), (7, 9), (8, 10), (9, 11), (10, 12),
                       (11, 13), (10, 14), (11, 15), (12, 14), (13, 15)]

    # Create a Plotly 3D scatter plot
    fig = go.Figure()

    all_values = ()
    idx = 0
    for keypoints, keypoints_dict, name in zip([keypoints_ground_truth, keypoints_hpe_truth],
                                               [keypoints_ground_truth, keypoints_hpe_truth],
                                               names):
        keypoints_array = np.array([v for v in keypoints_dict.values()])
        x, y, z = keypoints_array[:, 0], keypoints_array[:, 1], keypoints_array[:, 2]

        # Scatter plot for keypoints
        for kp_name, kp_x, kp_y, kp_z in zip(keypoints_dict.keys(), x, y, z):
            fig.add_trace(go.Scatter3d(
                x=[kp_x], y=[kp_y], z=[kp_z], mode='markers',
                marker=dict(color=colors[idx], size=5),
                text=[kp_name], textposition="top center",
                name=f'{name} {kp_name}'
            ))

        # Plot connections
        for connection in connections:
            x_vals = [x[connection[0]], x[connection[1]]]
            y_vals = [y[connection[0]], y[connection[1]]]
            z_vals = [z[connection[0]], z[connection[1]]]
            fig.add_trace(go.Scatter3d(
                x=x_vals, y=y_vals, z=z_vals, mode='lines',
                line=dict(color=colors[idx]),
                name=f'{name} Connection {connection[0]}-{connection[1]}'
            ))

        idx += 1

        # Combine x, y, z values into a single list
        all_values = all_values + tuple(x) + tuple(y) + tuple(z)

    # Find the minimum and maximum values
    min_value = min(all_values) - abs(min(all_values) * 0.1)
    max_value = max(all_values) + abs(max(all_values) * 0.1)

    # Update layout to set axis limits
    fig.update_layout(
        scene=dict(
            aspectmode='cube',
            xaxis=dict(title='X', range=[min_value, max_value]),
            yaxis=dict(title='Y', range=[min_value, max_value]),
            zaxis=dict(title='Z', range=[min_value, max_value])
        )
    )

    # Log the 3D scatter plot using WandB
    fig.show()


In [25]:
def align_procrustes(target, prediction):
    """
    Procrustes MPJPE: MPJPE after rigid alignment (scale, rotation, and translation),
    often referred to as "Protocol #2" in many papers.
    Based on the implementation from https://github.com/miraymen/3dpw-eval/blob/master/evaluate.py
    :param target: Ground truth 3D joint positions, shape [sample, joint, 3]
    :param prediction: Predicted 3D joint positions, shape [sample, joint, 3]
    :return gt_all: Ground truth 3D joint positions after alignment, shape [sample, joint, 3]
    :return pred_all: Predicted 3D joint positions after alignment, shape [sample, joint, 3]
    :return error_count: Error count of failed alignments
    """

    gt_all = []
    pred_all = []
    error_count = 0
    joint_number = target.shape[1]

    for (gt, pred) in (zip(target, prediction)):
        gt_raw = gt
        if not (np.sum(np.abs(pred)) == 0):
            transposed = False
            if pred.shape[0] != 3 and pred.shape[0] != 2:
                pred = pred.T
                gt = gt.T
                transposed = True
            assert (gt.shape[1] == pred.shape[1]), "The number of joints must match."

            # 1. Remove mean.
            muX = np.mean(pred, axis=1, keepdims=True)
            muY = np.mean(gt, axis=1, keepdims=True)
            X0 = pred - muX
            Y0 = gt - muY

            # 2. Compute variance of X1 used for scale.
            var1 = np.sum(X0 ** 2)

            # 3. The outer product of X1 and X2.
            K = X0.dot(Y0.T)

            # 4. Solution that Maximizes trace(R'K) is R=U*V', where U, V are singular vectors of K.
            try:
                U, s, Vh = np.linalg.svd(K)
            except np.linalg.LinAlgError:
                # print("SVD did not converge")
                error_count += 1
                continue

            V = Vh.T
            # Construct Z that fixes the orientation of R to get det(R)=1.
            Z = np.eye(U.shape[0])
            Z[-1, -1] *= np.sign(np.linalg.det(U.dot(V.T)))
            # Construct R.
            R = V.dot(Z.dot(U.T))

            # 5. Recover scale.
            scale = np.trace(R.dot(K)) / var1

            # 6. Recover translation.
            t = muY - scale * (R.dot(muX))

            # 7. Error:
            pred_hat = scale * R.dot(pred) + t

            if transposed:
                pred_hat = pred_hat.T

        else:
            pred_hat = np.tile(np.mean(gt, axis=0), (joint_number, 1))
            R = np.identity(3)

        gt_all.append(gt_raw)
        pred_all.append(pred_hat)

    gt_all = np.array(gt_all)
    pred_all = np.array(pred_all)

    if error_count > 0:
        print(f"Procrustes alignment failed {error_count} times")

    return gt_all, pred_all, error_count

def calculate_angle_point(joint_a, joint_b, joint_c):
    """
    Calculate the angle formed at joint_b by the lines connecting joint_a and joint_c
    :param joint_a: 3D coordinates of joint A, numpy array of shape [sample, joint, 3]
    :param joint_b: 3D coordinates of joint B (vertex of the angle), numpy array of shape [sample, joint, 3]
    :param joint_c: 3D coordinates of joint C, numpy array of shape [sample, joint, 3]
    :return: Angles in degrees, numpy array of shape [sample, joint]
    """
    # Check if any inputs need reshaping
    if len(joint_a.shape) == 1:
        joint_a = joint_a.reshape(-1, joint_a.shape[0], 3)
    if len(joint_b.shape) == 1:
        joint_b = joint_b.reshape(-1, joint_b.shape[0], 3)
    if len(joint_c.shape) == 1:
        joint_c = joint_c.reshape(-1, joint_c.shape[0], 3)

    v1 = joint_a - joint_b
    v2 = joint_c - joint_b
    v1_norm = v1 / np.linalg.norm(v1, axis=1, keepdims=True)
    v2_norm = v2 / np.linalg.norm(v2, axis=1, keepdims=True)
    dot_product = np.sum(v1_norm * v2_norm, axis=1)
    dot_product = np.clip(dot_product, -1.0, 1.0)  # Clip to ensure they are within the valid range for arccos [-1, 1]
    angles_rad = np.arccos(dot_product)
    angles_deg = np.degrees(angles_rad)
    return np.round(angles_deg, 2)


def orthogonal_projection(vector, normal):
    """
    Calculate the orthogonal projection of a vector onto a plane defined by a normal vector.
    :param vector: The vector to be projected, shape (batch, 3)
    :param normal: The normal vector of the plane, shape (batch, 3)
    :return: The orthogonal projection of the vector onto the plane, shape (batch, 3)
    """
    dot_product = np.einsum('ij,ij->i', vector, normal)
    normal_norm_sq = np.einsum('ij,ij->i', normal, normal)
    projection = vector - (dot_product / normal_norm_sq)[:, np.newaxis] * normal
    return projection


def get_vertical_axis_from_calibration(hip_left, hip_right, shoulder_left, shoulder_right):
    """
    Determine the vertical axis based on the positions of the hips and shoulders. Needs a relative neutral pose
    :param hip_left: numpy array of shape (3,), position of the left hip
    :param hip_right: numpy array of shape (3,), position of the right hip
    :param shoulder_left: numpy array of shape (3,), position of the left shoulder
    :param shoulder_right: numpy array of shape (3,), position of the right shoulder
    :return: normalized vertical axis vector
    """
    hip_mid = (hip_left + hip_right) / 2
    shoulder_mid = (shoulder_left + shoulder_right) / 2
    vertical_vector = shoulder_mid - hip_mid
    return vertical_vector / np.linalg.norm(vertical_vector)


def calculate_angle_vector(v1, v2):
    """
    Calculate the angle between two vectors.
    :param v1: First vector, numpy array of shape (batch, 3)
    :param v2: Second vector, numpy array of shape (batch, 3)
    :return: Angles in degrees, numpy array of shape (batch,)
    """
    # Check if v1 needs reshaping
    if len(v1.shape) == 1:
        v1 = v1.reshape(-1, 3)

    v1_norm = v1 / np.linalg.norm(v1, axis=1, keepdims=True)
    v2_norm = v2 / np.linalg.norm(v2, axis=1, keepdims=True)
    dot_product = np.einsum('ij,ij->i', v1_norm, v2_norm)
    dot_product = np.clip(dot_product, -1.0, 1.0)  # Clip to ensure they are within the valid range for arccos [-1, 1]
    angles_rad = np.arccos(dot_product)
    angles_deg = np.degrees(angles_rad)
    return np.round(angles_deg, 2)


def calculate_joint_angles(keypoints, Y_vector=np.array([0, 1, 0])):
    """
    Calculate various joint angles based on 3D keypoints.
    :param keypoints: Dictionary of 3D coordinates for each joint
    :param Y_vector: Reference vertical axis vector (default is the Y-axis)
    :return: Dictionary of calculated joint angles
    """
    angles = {}

    # Midpoints
    shoulder_mid = (keypoints['right_shoulder'] + keypoints['left_shoulder']) / 2
    hip_mid = (keypoints['right_hip'] + keypoints['left_hip']) / 2
    hip_mid = hip_mid.reshape(-1, 3)
    shoulder_mid = shoulder_mid.reshape(-1, 3)

    # Help vectors
    D_s = np.cross(hip_mid - shoulder_mid, keypoints['right_shoulder'] - keypoints['left_shoulder'])
    D_h = np.cross(Y_vector, keypoints['right_hip'] - keypoints['left_hip'])

    # Trunk angles
    angles['trunk_angle'] = 90 - calculate_angle_vector(hip_mid - shoulder_mid, D_h)
    angles['trunk_twist'] = calculate_angle_vector(
        orthogonal_projection(keypoints['left_hip'] - keypoints['right_hip'], shoulder_mid - hip_mid),
        orthogonal_projection(keypoints['right_shoulder'] - keypoints['left_shoulder'], shoulder_mid - hip_mid))
    angles['trunk_bend'] = calculate_angle_vector(Y_vector, orthogonal_projection(shoulder_mid - hip_mid, D_h))

    # Lower limb angles
    angles['knee_angle_l'] = calculate_angle_point(keypoints['left_hip'], hip_mid, keypoints['left_ankle']) # Really mid hip here?
    angles['ankle_angle_l'] = calculate_angle_point(keypoints['left_knee'], keypoints['left_ankle'], keypoints['left_foot_index'])
    angles['knee_angle_r'] = calculate_angle_point(keypoints['right_hip'], hip_mid, keypoints['right_ankle'])  # Really mid hip here?
    angles['ankle_angle_r'] = calculate_angle_point(keypoints['right_knee'], keypoints['right_ankle'], keypoints['right_foot_index'])

    # Upper limb angles
    angles['shoulder_angle_l'] = calculate_angle_vector(orthogonal_projection(keypoints['left_elbow'] - keypoints['left_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['left_elbow'] - keypoints['left_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_l'] = calculate_angle_point(keypoints['left_shoulder'], keypoints['left_elbow'], keypoints['left_wrist'])
    angles['shoulder_angle_r'] = calculate_angle_vector(orthogonal_projection(keypoints['right_elbow'] - keypoints['right_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['right_elbow'] - keypoints['right_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_r'] = calculate_angle_point(keypoints['right_shoulder'], keypoints['right_elbow'], keypoints['right_wrist'])

    return angles


def calculate_angle_error(target, prediction, Y_target=np.array([0, 1, 0]), Y_prediction=np.array([0, 1, 0]),
                          procrustes=False, calculate_r2=True, box=True):
    """
    Calculate the angle error between target and prediction joint positions.
    :param target: Ground truth 3D joint positions, shape [sample, joint, 3]
    :param prediction: Predicted 3D joint positions, shape [sample, joint, 3]
    :param Y_target: Reference vertical axis vector for target (default is the Y-axis)
    :param Y_prediction: Reference vertical axis vector for prediction (default is the Y-axis)
    :param procrustes: Whether to use Procrustes alignment
    :param r2: Whether to calculate R² value
    :param box: Whether to plot a box plot of the angle errors
    :return: Mean error, standard deviation of error, and R² value (if calculated)
    """
    if procrustes:
        target, prediction, error_count = align_procrustes(target, prediction)

    target_angles = calculate_joint_angles(target, Y_target)
    prediction_angles = calculate_joint_angles(prediction, Y_prediction)
    angle_names = list(target_angles.keys())

    target_angle_values = []
    prediction_angle_values = []
    for joint, angle in target_angles.items():
        target_angle_values.append(angle)
    for joint, angle in prediction_angles.items():
        prediction_angle_values.append(angle)

    angle_error = np.abs(np.array(target_angle_values) - np.array(prediction_angle_values))
    mean_error = np.mean(angle_error, axis=1)
    std_error = np.std(angle_error, axis=1)

    mean_dict = {angle: (mean_error[i]) for i, angle in enumerate(angle_names)}
    std_dict = {angle: (std_error[i]) for i, angle in enumerate(angle_names)}

    r2 = None
    if calculate_r2:
        r2 = plot_calculate_r2(target_angles, prediction_angles)

    if box:
        angle_error = {angle: (angle_error[i]) for i, angle in enumerate(angle_names)}
        plot_box(angle_error)
    return mean_dict, std_dict, r2


def plot_calculate_r2(target_angles_dict, prediction_angles_dict):
    """
    Plot R² values for the linear regression fit of target vs prediction angles for each metric and calculate R² value.
    :param target_angles_dict: Dictionary where keys are metric names and values are lists of ground truth angles
    :param prediction_angles_dict: Dictionary where keys are metric names and values are lists of predicted angles
    :return: Dictionary of R² values for each metric
    """
    r2_values = {}

    fig = go.Figure()

    colors = ['blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'brown', 'black', 'pink']
    symbols = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right', 'pentagon', 'hexagon']

    for i, metric in enumerate(target_angles_dict.keys()):
        target_angles = np.array(target_angles_dict[metric]).reshape(-1, 1)
        prediction_angles = np.array(prediction_angles_dict[metric])
        reg = LinearRegression().fit(target_angles, prediction_angles)
        prediction_line = reg.predict(target_angles)
        r2 = r2_score(prediction_angles, prediction_line)
        r2_values[metric] = r2

        # Scatter plot of the actual values with different colors and marker forms
        fig.add_trace(go.Scatter(
            x=target_angles.flatten(),
            y=prediction_angles,
            mode='markers',
            name=f'{metric} Actual values',
            marker=dict(color=colors[i % len(colors)], symbol=symbols[i % len(symbols)])
        ))

        # Linear regression line
        fig.add_trace(go.Scatter(
            x=target_angles.flatten(),
            y=prediction_line,
            mode='lines',
            name=f'{metric} Linear fit (R²={r2:.2f})',
            line=dict(color=colors[i % len(colors)], width=6)
        ))

    fig.update_layout(
        title='Prediction Angles vs Target Angles',
        xaxis_title='Target Angles',
        yaxis_title='Prediction Angles',
        legend_title='Legend',
        showlegend=True
    )

    fig.show()
    return r2_values


def plot_box(angle_error):
    """
    Plot different errors in a box plot.
    :param errors_dict: Dictionary where keys are metric names and values are lists of error values.
    """
    fig = go.Figure()

    # Different colors and symbols for different metrics
    colors = ['blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'brown', 'black', 'pink']
    symbols = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'triangle-left', 'triangle-right', 'pentagon', 'hexagon']

    for i, (metric, errors) in enumerate(angle_error.items()):
        fig.add_trace(go.Box(y=errors, name=metric, boxmean=True, marker_color=colors[i % len(colors)], marker_symbol=symbols[i % len(symbols)]))

    fig.update_layout(
        title='Error Metrics',
        xaxis_title='Metrics',
        yaxis_title='Error Values',
        showlegend=True
    )

    fig.show()


In [15]:
plot_3d_keypoints_validation(gt, pred, model_name)

In [26]:
angle = calculate_joint_angles(gt)

AxisError: axis 1 is out of bounds for array of dimension 1

In [28]:
    angles = {}

    # Midpoints
    shoulder_mid = (keypoints['right_shoulder'] + keypoints['left_shoulder']) / 2
    hip_mid = (keypoints['right_hip'] + keypoints['left_hip']) / 2
    hip_mid = hip_mid.reshape(-1, 3)
    shoulder_mid = shoulder_mid.reshape(-1, 3)

    # Help vectors
    D_s = np.cross(hip_mid - shoulder_mid, keypoints['right_shoulder'] - keypoints['left_shoulder'])
    D_h = np.cross(Y_vector, keypoints['right_hip'] - keypoints['left_hip'])

    # Trunk angles
    angles['trunk_angle'] = 90 - calculate_angle_vector(hip_mid - shoulder_mid, D_h)
    angles['trunk_twist'] = 180 - calculate_angle_vector(
        orthogonal_projection(keypoints['left_hip'] - keypoints['right_hip'], shoulder_mid - hip_mid),
        orthogonal_projection(keypoints['right_shoulder'] - keypoints['left_shoulder'], shoulder_mid - hip_mid))
    angles['trunk_bend'] = 90 - calculate_angle_vector(Y_vector, orthogonal_projection(shoulder_mid - hip_mid, D_h))

    # Lower limb angles
    angles['knee_angle_l'] = 90 + calculate_angle_point(keypoints['left_hip'], hip_mid, keypoints['left_ankle']) # Really mid hip here?
    angles['ankle_angle_l'] = calculate_angle_point(keypoints['left_knee'], keypoints['left_ankle'], keypoints['left_foot_index'])
    angles['knee_angle_r'] = 90 + calculate_angle_point(keypoints['right_hip'], hip_mid, keypoints['right_ankle'])  # Really mid hip here?
    angles['ankle_angle_r'] = calculate_angle_point(keypoints['right_knee'], keypoints['right_ankle'], keypoints['right_foot_index'])

    # Upper limb angles
    angles['shoulder_angle_l'] = calculate_angle_vector(orthogonal_projection(keypoints['left_elbow'] - keypoints['left_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['left_elbow'] - keypoints['left_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_l'] = calculate_angle_point(keypoints['left_shoulder'], keypoints['left_elbow'], keypoints['left_wrist'])
    angles['shoulder_angle_r'] = calculate_angle_vector(orthogonal_projection(keypoints['right_elbow'] - keypoints['right_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['right_elbow'] - keypoints['right_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_r'] = calculate_angle_point(keypoints['right_shoulder'], keypoints['right_elbow'], keypoints['right_wrist'])


NameError: name 'keypoints' is not defined

In [33]:
keypoints = gt
Y_vector=np.array([0, 1, 0])

In [35]:
    angles = {}

    # Midpoints
    shoulder_mid = (keypoints['right_shoulder'] + keypoints['left_shoulder']) / 2
    hip_mid = (keypoints['right_hip'] + keypoints['left_hip']) / 2
    hip_mid = hip_mid.reshape(-1, 3)
    shoulder_mid = shoulder_mid.reshape(-1, 3)

    # Help vectors
    D_s = np.cross(hip_mid - shoulder_mid, keypoints['right_shoulder'] - keypoints['left_shoulder'])
    D_h = np.cross(Y_vector, keypoints['right_hip'] - keypoints['left_hip'])

    # Trunk angles
    angles['trunk_angle'] = 90 - calculate_angle_vector(hip_mid - shoulder_mid, D_h)
    angles['trunk_twist'] = calculate_angle_vector(
        orthogonal_projection(keypoints['left_hip'] - keypoints['right_hip'], shoulder_mid - hip_mid),
        orthogonal_projection(keypoints['right_shoulder'] - keypoints['left_shoulder'], shoulder_mid - hip_mid))
    angles['trunk_bend'] = calculate_angle_vector(Y_vector, orthogonal_projection(shoulder_mid - hip_mid, D_h))

    # Lower limb angles
    angles['knee_angle_l'] = calculate_angle_point(keypoints['left_hip'], hip_mid, keypoints['left_ankle']) # Really mid hip here?
    angles['ankle_angle_l'] = calculate_angle_point(keypoints['left_knee'], keypoints['left_ankle'], keypoints['left_foot_index'])
    angles['knee_angle_r'] = calculate_angle_point(keypoints['right_hip'], hip_mid, keypoints['right_ankle'])  # Really mid hip here?
    angles['ankle_angle_r'] = calculate_angle_point(keypoints['right_knee'], keypoints['right_ankle'], keypoints['right_foot_index'])

    # Upper limb angles
    angles['shoulder_angle_l'] = calculate_angle_vector(orthogonal_projection(keypoints['left_elbow'] - keypoints['left_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['left_elbow'] - keypoints['left_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_l'] = calculate_angle_point(keypoints['left_shoulder'], keypoints['left_elbow'], keypoints['left_wrist'])
    angles['shoulder_angle_r'] = calculate_angle_vector(orthogonal_projection(keypoints['right_elbow'] - keypoints['right_shoulder'], np.cross(D_s, shoulder_mid - hip_mid)), shoulder_mid - hip_mid) * np.sign(np.dot(keypoints['right_elbow'] - keypoints['right_shoulder'], D_s.T))[1]  # Not sure if this is right
    angles['elbow_angle_r'] = calculate_angle_point(keypoints['right_shoulder'], keypoints['right_elbow'], keypoints['right_wrist'])

AxisError: axis 1 is out of bounds for array of dimension 1

In [36]:
a = (D_h.T)